In [ ]:
# 1. Carregamento e inspeção inicial

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# URL do dataset
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"

# Definição das colunas
feature_groups = ["mean", "se", "worst"]
base_features = [
    "radius", "texture", "perimeter", "area", "smoothness",
    "compactness", "concavity", "concave_points", "symmetry",
    "fractal_dimension"
]
feature_cols = [f"{b}_{g}" for g in feature_groups for b in base_features]
cols = ["id", "diagnosis"] + feature_cols

# Carregando o dataset
df = pd.read_csv(DATA_URL, header=None, names=cols)

# Mostrando informações básicas
print("Dimensões do dataset:", df.shape)
print("\nPrimeiras linhas:\n", df.head())

In [ ]:
# 2. Estatísticas descritivas básicas

variaveis = ["radius_mean", "texture_mean", "area_mean"]

for var in variaveis:
    print(f"\nVariável: {var}")
    print("Média:", df[var].mean())
    print("Mediana:", df[var].median())
    print("Desvio padrão:", df[var].std())
    print("Quartis:")
    print(df[var].quantile([0.25, 0.5, 0.75]))
    iqr = df[var].quantile(0.75) - df[var].quantile(0.25)
    print("IQR:", iqr)

In [ ]:
# 3. Análise por grupo

# Média de radius_mean por diagnóstico
print("\nMédia de radius_mean por diagnóstico:")
print(df.groupby("diagnosis")["radius_mean"].mean())

# Comparando média de area_worst
print("\nMédia de area_worst por diagnóstico:")
print(df.groupby("diagnosis")["area_worst"].mean())


In [ ]:
# 4. Distribuições e frequências

# Distribuição absoluta e percentual
print("\nDistribuição absoluta de diagnóstico:")
print(df["diagnosis"].value_counts())

print("\nDistribuição percentual de diagnóstico:")
print(df["diagnosis"].value_counts(normalize=True) * 100)

# Tabela combinada
tabela_freq = df["diagnosis"].value_counts().to_frame("n")
tabela_freq["%"] = df["diagnosis"].value_counts(normalize=True) * 100
print("\nTabela com número de casos e percentual:")
print(tabela_freq)

# Gráfico de barras
tabela_freq["n"].plot(kind="bar", color=["skyblue", "salmon"])
plt.title("Distribuição de Diagnóstico")
plt.xlabel("Diagnóstico")
plt.ylabel("Número de casos")
plt.show()


In [ ]:

# 5. Correlação e filtros

# Correlação entre radius_mean e perimeter_mean
cor = df["radius_mean"].corr(df["perimeter_mean"])
print("\nCorrelação radius_mean x perimeter_mean:", cor)

# Criando coluna binária para diagnóstico
df_corr = df.copy()
df_corr["diagnosis_num"] = df_corr["diagnosis"].map({"M": 1, "B": 0})

# Seleciona apenas colunas numéricas
numeric_df = df_corr.select_dtypes(include=["float64", "int64"])

# Variáveis mais correlacionadas com diagnóstico
corr_diag = numeric_df.corr()["diagnosis_num"].drop("diagnosis_num").abs()
print("\nTop 3 variáveis mais correlacionadas com diagnóstico:")
print(corr_diag.sort_values(ascending=False).head(3))

# 10 casos com maior radius_worst
print("\nTop 10 casos com maior radius_worst:")
print(df[["diagnosis", "radius_worst", "area_worst"]].nlargest(10, "radius_worst"))



In [ ]:
# 6. Visualizações gráficas

# Histograma de radius_mean separado por diagnóstico
plt.figure(figsize=(8,5))
sns.histplot(
    data=df,
    x="radius_mean",
    hue="diagnosis",
    kde=True,
    palette={"M": "red", "B": "blue"}
)
plt.title("Histograma de Radius Mean por Diagnóstico")
plt.show()
print('\n')

# Boxplot de area_mean comparando grupos
plt.figure(figsize=(8,5))
sns.boxplot(
    data=df,
    x="diagnosis",
    y="area_mean",
    hue="diagnosis",
    dodge=False,  # evita duplicação de boxes
    legend=False, # remove legenda repetida
    palette={"M": "red", "B": "blue"}
)
plt.title("Boxplot de Area Mean por Diagnóstico")
plt.show()
print('\n')

# Scatterplot radius_mean x area_mean
plt.figure(figsize=(8,5))
sns.scatterplot(
    data=df,
    x="radius_mean",
    y="area_mean",
    hue="diagnosis",
    palette={"M": "red", "B": "blue"}
)
plt.title("Scatterplot: Radius Mean x Area Mean")
plt.show()
print('\n')

# Heatmap de correlação
plt.figure(figsize=(6,5))
sns.heatmap(
    df[["radius_mean", "texture_mean", "perimeter_mean", "area_mean"]].corr(),
    annot=True, cmap="coolwarm", fmt=".2f"
)
plt.title("Heatmap de Correlação")
plt.show()

